In [1]:
from tkinter import *
from tkinter import messagebox
from tkinter.ttk import Combobox, Progressbar
import joblib
import pandas as pd
import random
import threading
import time

In [2]:
root = None
frame_color = '#1c1c1c'
icon = 'images/icon.ico'
loaded_objects = {}

In [3]:
class Main:
    def __init__(self):
        global frame_color
        win_height, win_width = 200, 300

        self.loading_root = Tk()
        self.screen_height = self.loading_root.winfo_screenheight()
        self.screen_width = self.loading_root.winfo_screenwidth()
        x_position = int((self.screen_width / 2) - (win_width / 2))
        y_position = int((self.screen_height / 2) - (win_height / 2))
        self.loading_root.geometry(f'{win_width}x{win_height}+{x_position}+{y_position}')

        self.loading_root.overrideredirect(True)
        self.loading_root.resizable(False, False)

        frame = Frame(self.loading_root, bg=frame_color)
        frame.pack(fill=BOTH, expand=True)

        self.progress_label = Label(frame, text="Starting...", bg=frame_color, fg='white', font=("Helvetica", 12))
        self.progress_label.pack(side=TOP, pady=50)

        self.progress_bar = Progressbar(frame, orient=HORIZONTAL, mode='determinate', length=250)
        self.progress_bar.pack(side=TOP, pady=10)

        self.thread = threading.Thread(target=self.load_data, daemon=True).start()

        self.loading_root.mainloop()

    def root_setup(self):
        global root, frame_color,icon
        win_height, win_width = 700, 600

        root = Tk()
        x_position = int((self.screen_width / 2) - (win_width / 2))
        y_position = int((self.screen_height / 2) - (win_height / 2))
        root.geometry(f'{win_width}x{win_height}+{x_position}+{y_position}')

        root.title("Emotion Engine")
        root.iconbitmap(icon)
        MainFrame()
        root.mainloop()

    def load_data(self):
        global loaded_objects
        tasks = [
            ("Loading model", 'model/classifier.joblib', 'model'),
            ("Loading emotion encoder", 'encoders/emotion_encoder.joblib', 'emotion_encoder'),
            ("Loading gender encoder", 'encoders/gender_encoder.joblib', 'gender_encoder'),
            ("Loading platform encoder", 'encoders/platform_encoder.joblib', 'platform_encoder'),
            ("Loading age scaler", 'scalers/Age_scaler.joblib', 'Age_scaler'),
            ("Loading usage scaler", 'scalers/Usage_scaler.joblib', 'Usage_scaler'),
            ("Loading posts scaler", 'scalers/Posts_scaler.joblib', 'Posts_scaler'),
            ("Loading likes scaler", 'scalers/Likes_scaler.joblib', 'Likes_scaler'),
            ("Loading comments scaler", 'scalers/Comments_scaler.joblib', 'Comments_scaler'),
            ("Loading messages scaler", 'scalers/Messages_scaler.joblib', 'Messages_scaler')
        ]

        total_tasks = len(tasks)
        for i, (task, filepath, var_name) in enumerate(tasks):
            try:
                loaded_objects[var_name] = joblib.load(filepath)
            except Exception as e:
                messagebox.showerror(title="Error", message=str(e))
                self.loading_root.destroy()
                exit(1)

            progress = int((i + 1) / total_tasks * 100)
            time.sleep(0.01)
            self.update_progress(task, progress)

        self.update_progress("Load complete", 100)
        time.sleep(0.5)
        self.loading_root.after(0, self.show_main_window)

    def update_progress(self, task, progress):
        self.progress_label.config(text=task)
        self.progress_bar['value'] = progress
        if progress == 100:
            self.progress_label.config(text="Load complete")

    def show_main_window(self):
        self.loading_root.destroy()
        self.root_setup()



In [4]:
class MainFrame:
    def __init__(self):
        global frame_color
        self.font_label = ("Helvetica", 12)
        self.font_entry = ("Helvetica", 10)
        self.font_color = 'white'

        self.gender = StringVar(value="Male")

        self.main_frame = Frame(root, bg=frame_color)
        self.main_frame.pack(fill='both', expand=True)
        # -------------------------------------------------------------------------------------------------------------
        self.feature_frame = Frame(self.main_frame, bg=frame_color, bd=2, relief='flat', highlightbackground="grey",
                                   highlightcolor="grey", highlightthickness=1)
        self.feature_frame.pack(pady=10)
        # -------------------------------------------------------------------------------------------------------------
        Label(self.feature_frame, text="Features", bg=frame_color, fg=self.font_color, height=2, anchor=NW,
              font=('Helvetica', 15), ).grid(row=0, column=0, sticky=NW)
        # -------------------------------------------------------------------------------------------------------------
        self.age = self.label_maker(self.feature_frame, "Age", 1)
        # -------------------------------------------------------------------------------------------------------------
        Label(self.feature_frame, text="Gender", font=self.font_label, bg=frame_color, fg='white').grid(row=2, column=0,
                                                                                                        padx=10, pady=5,
                                                                                                        sticky=W)
        gender_frame = Frame(self.feature_frame, bg=frame_color)
        gender_frame.grid(row=2, column=1, padx=10, pady=5)

        self.rb1 = Radiobutton(gender_frame, text="Male", variable=self.gender, value="Male",
                               font=('Impact', 15), compound='left', bd=3, relief=SUNKEN, anchor="w",
                               command=self.rb1_button, indicatoron=0)
        self.rb1.pack(fill="both", side=LEFT)

        self.rb2 = Radiobutton(gender_frame, text="Female", variable=self.gender, value="Female", font=('Impact', 15),
                               compound='left', bd=5, relief=RAISED, anchor="w", command=self.rb2_button, indicatoron=0)
        self.rb2.pack(fill="both", side=LEFT)
        # -------------------------------------------------------------------------------------------------------------
        Label(self.feature_frame, text="Platform", font=self.font_label, bg=frame_color, fg='white').grid(row=3,
                                                                                                          column=0,
                                                                                                          padx=10,
                                                                                                          pady=5,
                                                                                                          sticky=W)
        platform_list = ("Instagram", "Twitter", "Facebook", "LinkedIn", "Snapchat", "Telegram")
        self.platform_dropdown = Combobox(self.feature_frame, font=self.font_entry, values=platform_list,state='readonly')
        self.platform_dropdown.grid(row=3, column=1, padx=10, pady=5)
        self.platform_dropdown.set("Instagram")
        # -------------------------------------------------------------------------------------------------------------
        self.comment = self.label_maker(self.feature_frame, "Comments Received Per Day", 4)
        # -------------------------------------------------------------------------------------------------------------
        self.like = self.label_maker(self.feature_frame, "Likes Received Per Day", 5)
        # -------------------------------------------------------------------------------------------------------------
        self.message = self.label_maker(self.feature_frame, "Messages Sent Per Day", 6)
        # -------------------------------------------------------------------------------------------------------------
        self.post = self.label_maker(self.feature_frame, "Posts Per Day", 7)
        # -------------------------------------------------------------------------------------------------------------
        self.usage = self.label_maker(self.feature_frame, "Daily Usage Time (minutes)", 8)
        # -------------------------------------------------------------------------------------------------------------

        Prediction_frame = Frame(self.main_frame, bg=frame_color, bd=2, relief='flat', highlightbackground="grey",
                                 highlightcolor="grey", highlightthickness=1)
        Prediction_frame.pack()
        # -------------------------------------------------------------------------------------------------------------

        buttons_frame = Frame(Prediction_frame, bg=frame_color)
        buttons_frame.pack(expand=True, side=TOP, anchor="n", padx=50)
        # -------------------------------------------------------------------------------------------------------------

        self.reset_button = Button(buttons_frame, text="Clear", command=self.clear, font=self.font_label)
        self.reset_button.grid(row=0, column=0, padx=5, pady=5, sticky="NSEW")
        # -------------------------------------------------------------------------------------------------------------

        self.predict_button = Button(buttons_frame, text="Predict", command=self.predict, font=self.font_label)
        self.predict_button.grid(row=0, column=1, padx=5, pady=5, sticky="NSEW")
        # -------------------------------------------------------------------------------------------------------------

        self.random_button = Button(buttons_frame, text="Random fill", command=self.random_fill, font=self.font_label)
        self.random_button.grid(row=0, column=2, padx=5, pady=5, sticky="NSEW")
        # -------------------------------------------------------------------------------------------------------------

        self.predict_label = Label(Prediction_frame, text="", bg=frame_color, height=2, fg=self.font_color,
                                   font=('Helvetica', 15))
        self.predict_label.pack(pady=10, side=TOP, fill='x', anchor=E)

    def rb1_button(self):
        self.rb1.config(relief=SUNKEN)
        self.rb2.config(relief=RAISED)

    def rb2_button(self):
        self.rb2.config(relief=SUNKEN)
        self.rb1.config(relief=RAISED)

    def clear(self):
        for entry in [self.age, self.comment, self.like, self.message, self.post, self.usage]:
            entry.delete(0, END)
        self.gender.set("Male")
        self.platform_dropdown.set("Instagram")
        self.predict_label.config(text="")

    def predict(self):
        global loaded_objects

        try:
            age = int(self.age.get())
            comments = int(self.comment.get())
            likes = int(self.like.get())
            messages = int(self.message.get())
            posts = int(self.post.get())
            usage = int(self.usage.get())
        except ValueError:
            self.predict_label.config(text="Only Enter Integer Numbers")
            return

        gender = loaded_objects['gender_encoder'].transform([self.gender.get()])[0]
        platform = loaded_objects['platform_encoder'].transform([self.platform_dropdown.get()])[0]

        df = pd.DataFrame({
            'Age': [age],
            'Gender': [gender],
            'Platform': [platform],
            'Comments_Received_Per_Day': [comments],
            'Likes_Received_Per_Day': [likes],
            'Messages_Sent_Per_Day': [messages],
            'Posts_Per_Day': [posts],
            'Daily_Usage_Time (minutes)': [usage]
        })

        df['Age'] = loaded_objects['Age_scaler'].transform(df[['Age']])
        df['Daily_Usage_Time (minutes)'] = loaded_objects['Usage_scaler'].transform(df[['Daily_Usage_Time (minutes)']])
        df['Posts_Per_Day'] = loaded_objects['Posts_scaler'].transform(df[['Posts_Per_Day']])
        df['Likes_Received_Per_Day'] = loaded_objects['Likes_scaler'].transform(df[['Likes_Received_Per_Day']])
        df['Comments_Received_Per_Day'] = loaded_objects['Comments_scaler'].transform(df[['Comments_Received_Per_Day']])
        df['Messages_Sent_Per_Day'] = loaded_objects['Messages_scaler'].transform(df[['Messages_Sent_Per_Day']])

        prediction = loaded_objects['model'].predict(df)[0]
        emotion = loaded_objects['emotion_encoder'].inverse_transform([prediction])[0]
        self.predict_label.config(text=f"Predicted Emotion: {emotion}")

    def random_fill(self):
        self.clear()
        self.age.insert(0, random.randint(1, 100))
        self.gender.set(random.choice(["Male", "Female"]))
        self.platform_dropdown.set(
            random.choice(["Instagram", "Twitter", "Facebook", "LinkedIn", "Snapchat", "Telegram"]))
        self.comment.insert(0, random.randint(0, 100))
        self.like.insert(0, random.randint(0, 100))
        self.message.insert(0, random.randint(0, 100))
        self.post.insert(0, random.randint(0, 100))
        self.usage.insert(0, random.randint(0, 1000))

    def label_maker(self, frame, label_text, row_number):
        global frame_color
        Label(frame, text=label_text, font=self.font_label, bg=frame_color, fg=self.font_color).grid(
            row=row_number, column=0,
            padx=10, pady=5,
            sticky=W)
        temp = Entry(frame, font=self.font_entry)
        temp.grid(row=row_number, column=1, padx=10, pady=5)
        return temp

In [ ]:
if __name__ == '__main__':
    Main()
else:
    messagebox.showerror(title='ERROR #1', message='You can not Run this file from another file')